In [6]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
from Utils.rflocate import RFMapList, load_rfmap, load_rf_tc, save_rf_tc
from Utils.rflocate.workflow import analyze_rf_file, save_rf_unit_lists
from Utils.tc_preparation import prepare_rf_comparison
from Utils.rflocate.plotting import rf_unit_plot_data, rf_population_counts, plot_rf_unit, plot_rf_population, export_rf_units
from scipy.ndimage import gaussian_filter
from Utils.plotting import apply_light_plot_style

apply_light_plot_style()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# base_dir = Path("/mnt/senzailab/Kai/#Recording/m20")
date = 260925
sessionID = 4
probes = ("A",)
is_rotation = False

rf_type = "excitatory"  # Plot selection; excitatory and inhibitory are both detected.
rf_time_range = (0.0, 0.2)
max_zero_bins = 2
rf_only: bool = True  # Sum only rows containing 2-D RF; False sums every row.
collapse_from_2d = False  # True projects the already computed 2-D RF; no separate 1-D detection.
is_save = True


In [8]:
# rf_session = base_dir / str(date) / f"{date}_{sessionID}"
rf_session = Path(r"R:\Basic_Sciences\Phys\SenzaiLab\Aparna\Data_by_mouse\m002\small-spread-neuropixel-config\2026-09-25\ad_session_2026_09_25\sessions\session_04" )
rf_dir = rf_session / "data/rfmapping/good/-100_400_1ms"
prefix = "rotation_30" if is_rotation else "regular"

rf_analyses_by_probe = {}
for probe in probes:
    rf_source = rf_dir / f"Probe{probe}" / f"{prefix}_unitsSpikeCounts_{date}_{sessionID}.rfmap"
    print(rf_source)
    rf_analyses_by_probe[probe] = analyze_rf_file(
        rf_source, probe=probe, rf_type="both", time_range_s=rf_time_range,
        max_zero_bins=max_zero_bins,
        collapse_from_2d=collapse_from_2d, save_results=is_save,
    )
    for detection_type, result in rf_analyses_by_probe[probe].items():
        tc_name = f"{rf_source.stem}_Probe{probe}"
        if rf_only and detection_type == "inhibitory":
            tc_name += "_inhibitory"
        tc_name += "_1d_rfonly.csv" if rf_only else "_1d.csv"
        tc_path = rf_session / "data" / tc_name
        if not tc_path.exists():
            raw_tc = load_rfmap(rf_source)
            if rf_only:
                raw_tc = RFMapList(
                    [raw_tc.by_unit_id(unit_id) for unit_id in result["result_2d"].unit_ids],
                    raw_tc.source_path,
                )
            tc_maps = raw_tc.sum(*rf_time_range, show_progress=False).sum_to_1d(
                axis="x", rf_only=rf_only,
                detected_rf=result["result_2d"] if rf_only else None,
            )
            save_rf_tc(tc_maps, tc_path, axis="x")
        result["tc_1d"] = load_rf_tc(tc_path)
        print(f"1D response CSV: {tc_path}")
        if is_save and not is_rotation:
            comparison_path = rf_session / "data/tc_comparison" / f"rf_{detection_type}_x_2d{'_rfonly' if rf_only else ''}_Probe{probe}.csv"
            prepare_rf_comparison(
                rf_source, comparison_path, projection_path=tc_path,
                detected_rf_path=result["output_paths"]["result_2d"],
                time_range_s=rf_time_range, rf_only=rf_only,
                unit_prefix=f"{base_dir.name}:{date}:{probe}",
            )
            print(f"Comparison CSV: {comparison_path}")
        print(
            f"Probe{probe} {detection_type}: kept {result['summed'].n_units} units; "
            f"2D RF {len(result['units_with_rf'])}; 1D RF {len(result['units_with_rf_1d'])}"
        )

if is_save:
    list_dir = rf_dir / f"Probe{probes[0]}" if len(probes) == 1 else rf_dir
    save_rf_unit_lists(rf_analyses_by_probe, list_dir)
    print(f"Unit lists: {list_dir}")


R:\Basic_Sciences\Phys\SenzaiLab\Aparna\Data_by_mouse\m002\small-spread-neuropixel-config\2026-09-25\ad_session_2026_09_25\sessions\session_04\data\rfmapping\good\-100_400_1ms\ProbeA\regular_unitsSpikeCounts_260925_4.rfmap
1D response CSV: R:\Basic_Sciences\Phys\SenzaiLab\Aparna\Data_by_mouse\m002\small-spread-neuropixel-config\2026-09-25\ad_session_2026_09_25\sessions\session_04\data\regular_unitsSpikeCounts_260925_4_ProbeA_1d_rfonly.csv
Comparison CSV: R:\Basic_Sciences\Phys\SenzaiLab\Aparna\Data_by_mouse\m002\small-spread-neuropixel-config\2026-09-25\ad_session_2026_09_25\sessions\session_04\data\tc_comparison\rf_excitatory_x_2d_rfonly_ProbeA.csv
ProbeA excitatory: kept 91 units; 2D RF 33; 1D RF 32
1D response CSV: R:\Basic_Sciences\Phys\SenzaiLab\Aparna\Data_by_mouse\m002\small-spread-neuropixel-config\2026-09-25\ad_session_2026_09_25\sessions\session_04\data\regular_unitsSpikeCounts_260925_4_ProbeA_inhibitory_1d_rfonly.csv
Comparison CSV: R:\Basic_Sciences\Phys\SenzaiLab\Aparna\Da

### Single-unit preview

Preview a unit from the first configured probe. Change only `unit_id` below.


In [ ]:
unit_id = 286
probe = probes[0]
unit_plot_data = rf_unit_plot_data(rf_analyses_by_probe[probe][rf_type], unit_id, rf_only=rf_only)
fig, axes = plot_rf_unit(
    unit_plot_data,
    title=f"Probe{probe} · Unit {unit_id} · {rf_type} RF",
)


### Population RF

Each probe, all probes combined, and smoothed display maps. Each figure shows 2D and 1D RF areas and centers.


In [ ]:
population_counts = rf_population_counts(rf_analyses_by_probe, rf_type=rf_type)
population_counts["all_smoothed"] = {
    key: gaussian_filter(values.astype(float), sigma=1.0, mode="nearest")
    for key, values in population_counts["all"].items()
}
population_figures = plot_rf_population(
    population_counts, rf_type=rf_type,
    output_dir=rf_session / "data/rfmapping" if is_save else None,
)


### Export unit figures

Save one combined 2D/1D figure per unit as PNG and SVG, without inline output. This cell writes figures only when `is_save=True`.


In [ ]:
if is_save:
    exported_units = export_rf_units(
        rf_analyses_by_probe, rf_session / "data/rfmapping", rf_type=rf_type, rf_only=rf_only,
    )
    print(f"Exported {exported_units} unit figures.")
